In [1]:
import pandas as pd
import glob

In [2]:
# Find the latest collected file
file = sorted(glob.glob("../data/raw/jobs_2*.csv"))[-1]
df = pd.read_csv(file)

print("File:", file)
print("Rows, Columns:", df.shape)
df.head()

File: ../data/raw\jobs_2026-09-24.csv
Rows, Columns: (2314, 16)


,job_id,search_role,title,company,location,location_area,category,contract_time,contract_type,salary_min,salary_max,salary_is_predicted,created,description,url,collected_on
0,5893922940,data analyst,Data Analyst,IVTEX Corporate Solutions Private Limited,India,India,IT Jobs,full_time,NaN,NaN,NaN,0,2026-09-22T17:36:09Z,Company Description Ivtex Corporate Solutions ...,https://www.adzuna.in/land/ad/5893922940?se=bo...,2026-09-24
1,5874134576,data analyst,Data Analyst,KRFS,"Karnataka, India",India > Karnataka,IT Jobs,NaN,NaN,NaN,NaN,0,2026-09-07T18:37:51Z,We are looking for a Data Analyst who can coll...,https://www.adzuna.in/land/ad/5874134576?se=bo...,2026-09-24
2,5868794225,data analyst,Data Analyst,Indofast Energy,India,India,IT Jobs,NaN,NaN,NaN,NaN,0,2026-09-03T16:50:23Z,About the Role : We are looking for a highly a...,https://www.adzuna.in/land/ad/5868794225?se=bo...,2026-09-24
3,5876645051,data analyst,Data Analyst,Cornerstone OnDemand,"Pune, Maharashtra",India > Maharashtra > Pune,IT Jobs,full_time,NaN,NaN,NaN,0,2026-09-09T10:19:37Z,"This job is with Cornerstone OnDemand, an incl...",https://www.adzuna.in/land/ad/5876645051?se=bo...,2026-09-24
4,5867036037,data analyst,Data Analyst,Daice Labs,India,India,IT Jobs,full_time,NaN,NaN,NaN,0,2026-09-02T16:42:41Z,Job Title: Data Analyst Location: India (Remot...,https://www.adzuna.in/land/ad/5867036037?se=bo...,2026-09-24


In [3]:
# 1. Percentage of missing values in each column
print("Missing values (%):")
print((df.isna().mean() * 100).round(1))

Missing values (%):
job_id                  0.0
search_role             0.0
title                   0.0
company                 0.7
location                0.0
location_area           0.0
category                0.0
contract_time          36.2
contract_type          93.3
salary_min             74.5
salary_max             74.5
salary_is_predicted     0.0
created                 0.0
description             0.0
url                     0.0
collected_on            0.0
dtype: float64


In [4]:
# 2. Top 10 locations
print("\nTop 10 locations:")
print(df["location"].value_counts().head(10))


Top 10 locations:
location
India                   665
Bangalore, Karnataka    640
Hyderabad, Telangana    244
Pune, Maharashtra       153
Mumbai, Maharashtra     126
Chennai, Tamil Nadu     109
Noida, Ghaziabad         61
Delhi, India             34
Ahmedabad, Gujarat       30
Gurgaon, Haryana         28
Name: count, dtype: int64


In [5]:
# 3. How many salaries are real vs. estimated by Adzuna
print("\nSalary predicted (1 = estimated, 0 = real):")
print(df["salary_is_predicted"].value_counts())


Salary predicted (1 = estimated, 0 = real):
salary_is_predicted
0    2314
Name: count, dtype: int64


In [6]:
salary_df = df.dropna(subset=["salary_min"])

print("Jobs with salary:", len(salary_df))
print(salary_df[["salary_min", "salary_max"]].describe().round(0))

print("\nSample rows:")
salary_df[["title", "location", "salary_min", "salary_max"]].sample(10, random_state=1)

Jobs with salary: 591
       salary_min  salary_max
count       591.0       591.0
mean     846677.0   1658261.0
std      778653.0   1130276.0
min           0.0       144.0
25%      300000.0    800000.0
50%      600000.0   1500000.0
75%     1100000.0   2200000.0
max     4500000.0   6000000.0

Sample rows:


,title,location,salary_min,salary_max
1892,Machine Learning Engineeer - Remote,India,208000.0,312000.0
985,Data Scientist,"Mumbai, Maharashtra",300000.0,800000.0
107,Data Analyst,India,500000.0,600000.0
1620,Machine Learning Engineer,India,0.0,900000.0
271,Sr. Data Analyst,"Bangalore, Karnataka",600000.0,1500000.0
507,Business Analyst,"Bangalore, Karnataka",0.0,500000.0
52,Data Analyst,India,264000.0,276000.0
647,Business Analyst,"Chembur, Mumbai",300000.0,600000.0
1072,Data Scientist,"Thiruvananthapuram, Kerala",2000000.0,3000000.0
1034,Data Scientist,"Bangalore, Karnataka",500000.0,1500000.0


In [7]:
# Rows where salary_min is 0
zero_min = salary_df[salary_df["salary_min"] == 0]
print("Rows with salary_min = 0:", len(zero_min))

# Rows with suspiciously low salary_max (under 1 lakh per year)
low = salary_df[salary_df["salary_max"] < 100000]
print("Rows with salary_max under 1 lakh:", len(low))
low[["title", "location", "salary_min", "salary_max"]]

Rows with salary_min = 0: 50
Rows with salary_max under 1 lakh: 7


,title,location,salary_min,salary_max
116,Data Analyst,"Noida, Ghaziabad",20000.0,25000.0
260,Intern - Data Analyst,"Gulmohar Park, South Delhi",20000.0,20000.0
364,Data Analyst I,India,22000.0,36000.0
522,Business Analyst,"Gurgaon, Haryana",12.0,144.0
1070,Data Scientist,India,3000.0,4000.0
1119,Data Scientist,"Surat, Gujarat",30000.0,30000.0
2223,Senior AI Engineer,"Bangalore, Karnataka",1000.0,1000.0


In [8]:
clean = df.copy()

# Rule 1: values between 10,000 and 1 lakh are monthly -> convert to yearly
monthly = clean["salary_max"].between(10000, 99999)
clean.loc[monthly, ["salary_min", "salary_max"]] *= 12

# Rule 2: values under 10,000 are unreliable -> remove salary
junk = clean["salary_max"] < 10000
clean.loc[junk, ["salary_min", "salary_max"]] = None

# Rule 3: salary_min of 0 means "not given" -> mark as unknown
clean.loc[clean["salary_min"] == 0, "salary_min"] = None

# Average salary only where both min and max are known
clean["salary_avg"] = (clean["salary_min"] + clean["salary_max"]) / 2

# Show salaries in lakhs (easier to read)
for col in ["salary_min", "salary_max", "salary_avg"]:
    clean[col + "_lakh"] = (clean[col] / 100000).round(2)

print("Monthly rows converted:", monthly.sum())
print("Junk rows removed:", junk.sum())
print("Jobs with average salary:", clean["salary_avg"].notna().sum())
print()
print(clean[["salary_min_lakh", "salary_max_lakh", "salary_avg_lakh"]].describe().round(2))

Monthly rows converted: 4
Junk rows removed: 3
Jobs with average salary: 538

       salary_min_lakh  salary_max_lakh  salary_avg_lakh
count           538.00           588.00           538.00
mean              9.32            16.69            13.26
std               7.65            11.24             9.00
min               0.22             1.00             0.95
25%               4.00             8.00             6.50
50%               7.00            15.00            11.30
75%              12.00            22.00            17.50
max              45.00            60.00            50.00


In [9]:
clean = df.copy()

# Rule 1: values between 10,000 and 1 lakh are monthly -> convert to yearly
monthly = clean["salary_max"].between(10000, 99999)
clean.loc[monthly, ["salary_min", "salary_max"]] *= 12

# Rule 2: values under 10,000 are unreliable -> remove salary
junk = clean["salary_max"] < 10000
clean.loc[junk, ["salary_min", "salary_max"]] = None

# Rule 3: salary_min of 0 means "not given" -> mark as unknown
clean.loc[clean["salary_min"] == 0, "salary_min"] = None

# Average salary only where both min and max are known
clean["salary_avg"] = (clean["salary_min"] + clean["salary_max"]) / 2

# Show salaries in lakhs (easier to read)
for col in ["salary_min", "salary_max", "salary_avg"]:
    clean[col + "_lakh"] = (clean[col] / 100000).round(2)

print("Monthly rows converted:", monthly.sum())
print("Junk rows removed:", junk.sum())
print("Jobs with average salary:", clean["salary_avg"].notna().sum())
print()
print(clean[["salary_min_lakh", "salary_max_lakh", "salary_avg_lakh"]].describe().round(2))

Monthly rows converted: 4
Junk rows removed: 3
Jobs with average salary: 538

       salary_min_lakh  salary_max_lakh  salary_avg_lakh
count           538.00           588.00           538.00
mean              9.32            16.69            13.26
std               7.65            11.24             9.00
min               0.22             1.00             0.95
25%               4.00             8.00             6.50
50%               7.00            15.00            11.30
75%              12.00            22.00            17.50
max              45.00            60.00            50.00


In [10]:
print(clean["location_area"].value_counts().head(25))

location_area
India                                            665
India > Karnataka > Bangalore                    640
India > Telangana > Hyderabad                    244
India > Maharashtra > Pune                       153
India > Maharashtra > Mumbai                     126
India > Tamil Nadu > Chennai                     109
India > Uttar Pradesh > Ghaziabad > Noida         61
India > Delhi                                     34
India > Gujarat > Ahmedabad                       30
India > Haryana > Gurgaon                         28
India > Maharashtra > Navi Mumbai                 20
India > West Bengal > Kolkata                     18
India > Tamil Nadu > Coimbatore                   13
India > Rajasthan > Jaipur                        13
India > Karnataka > Bangalore > Richmond Town     11
India > Gujarat > Vadodara                        10
India > Kerala > Ernakulam > Kochi                 8
India > Punjab > Mohali                            6
India > Karnataka > Bangalore > 

In [11]:
def split_location(area):
    parts = [p.strip() for p in str(area).split(">")]
    state = parts[1] if len(parts) > 1 else "Unspecified"
    city = parts[2] if len(parts) > 2 else None

    # Special cases
    if state == "Delhi":
        city = "Delhi"
    if city == "Ghaziabad" and len(parts) > 3 and parts[3] == "Noida":
        city = "Noida"
    if city is None:
        city = "India (unspecified)" if state == "Unspecified" else state

    # Standardise alternate city names
    aliases = {"Ernakulam": "Kochi", "Bengaluru": "Bangalore", "Gurugram": "Gurgaon"}
    city = aliases.get(city, city)
    return city, state

clean[["city", "state"]] = clean["location_area"].apply(
    lambda a: pd.Series(split_location(a))
)

print("Number of unique cities:", clean["city"].nunique())
print()
print(clean["city"].value_counts().head(15))

Number of unique cities: 52

city
India (unspecified)    665
Bangalore              665
Hyderabad              251
Pune                   156
Mumbai                 138
Chennai                113
Noida                   61
Delhi                   41
Ahmedabad               33
Gurgaon                 30
Navi Mumbai             20
Kolkata                 18
Coimbatore              13
Jaipur                  13
Kochi                   12
Name: count, dtype: int64


In [12]:
print("Unique titles:", clean["title"].nunique())
print()
print(clean["title"].value_counts().head(40))


Unique titles: 672

title
Business Analyst                                             307
Data Scientist                                               228
Data Analyst                                                 208
Data Engineer                                                201
AI Engineer                                                  131
Machine Learning Engineer                                     76
Senior AI Engineer                                            54
Senior Data Analyst                                           47
Senior Machine Learning Engineer                              42
Lead AI Engineer                                              35
Senior Data Engineer                                          22
Principal AI Engineer                                         16
Azure Data Engineer                                           11
Staff Machine Learning Engineer                               11
Agentic AI Engineer                                           11

In [13]:
import re

def get_role(title):
    t = str(title).lower()
    if "machine learning" in t or re.search(r"\bml\b", t):
        return "ML Engineer"
    if re.search(r"\b(ai|gen ai|genai|llm)\b", t) or "artificial intelligence" in t:
        return "AI Engineer"
    if "data scien" in t:
        return "Data Scientist"
    if "data engineer" in t:
        return "Data Engineer"
    if "business analyst" in t:
        return "Business Analyst"
    if "analyst" in t or "analytics" in t:
        return "Data Analyst"
    return "Other"

def get_seniority(title):
    t = str(title).lower()
    if re.search(r"\b(intern|internship|trainee)\b", t):
        return "Intern"
    if re.search(r"\b(lead|principal|staff|manager|head|director|architect)\b", t):
        return "Lead / Principal"
    if re.search(r"\b(senior|sr)\b", t):
        return "Senior"
    if re.search(r"\b(junior|jr|entry|fresher|graduate|associate)\b", t) or t.strip().endswith(" i"):
        return "Junior"
    return "Not specified"

clean["role"] = clean["title"].apply(get_role)
clean["seniority"] = clean["title"].apply(get_seniority)

print(clean["role"].value_counts())
print()
print(clean["seniority"].value_counts())
print()
print("Sample of 'Other' titles:")
print(clean.loc[clean["role"] == "Other", "title"].value_counts().head(15))

role
AI Engineer         426
Data Analyst        403
Data Engineer       375
Business Analyst    348
ML Engineer         328
Data Scientist      306
Other               128
Name: count, dtype: int64

seniority
Not specified       1638
Senior               350
Lead / Principal     280
Junior                26
Intern                20
Name: count, dtype: int64

Sample of 'Other' titles:
title
Data Architect                                              4
Senior Software Engineer_AI Assistant                       4
MLOps Databricks Engineer                                   3
Automation Engineer                                         2
Product Manager                                             2
Back End Developer                                          2
Software Associate                                          2
Python with Django - Python with Django L3 (5.1-7 years)    2
Assistant Professor                                         2
Staff Software Engineer                         

In [14]:
def get_role(title):
    t = str(title).lower().replace("_", " ")
    t = re.sub(r"\bal\b", "ai", t)   # fix "Al Engineer" typo

    if "machine learning" in t or re.search(r"\b(ml|mlops)\b", t):
        return "ML Engineer"
    if re.search(r"\b(ai|gen ai|genai|llm)\b", t) or "artificial intelligence" in t:
        return "AI Engineer"
    if "data scien" in t or "applied scien" in t:
        return "Data Scientist"
    if "data engineer" in t or "data architect" in t or re.search(r"\betl\b", t):
        return "Data Engineer"
    if "business analyst" in t:
        return "Business Analyst"
    if "analyst" in t or "analytics" in t or "business intelligence" in t or re.search(r"\bbi\b", t):
        return "Data Analyst"
    return "Other"

clean["role"] = clean["title"].apply(get_role)

print(clean["role"].value_counts())
print()
print("Remaining 'Other' titles:")
print(clean.loc[clean["role"] == "Other", "title"].value_counts().head(15))

role
AI Engineer         431
Data Analyst        405
Data Engineer       381
Business Analyst    348
ML Engineer         335
Data Scientist      309
Other               105
Name: count, dtype: int64

Remaining 'Other' titles:
title
Automation Engineer                                         2
Product Manager                                             2
Back End Developer                                          2
Software Associate                                          2
Python with Django - Python with Django L3 (5.1-7 years)    2
Assistant Professor                                         2
Staff Software Engineer                                     2
Project Manager                                             2
GL Data & System Care – Manager                             1
Staff Backend Engineer                                      1
Information Technology Internal Auditor                     1
Liquidity reporting Compliance                              1
Head Customer Success [T

In [15]:
def get_role(title):
    # Make the title lowercase so "Data Analyst" and "data analyst" match
    t = str(title).lower()

    # Replace symbols with spaces, so "Engineer_AI" becomes "engineer ai"
    for symbol in ["_", "/", "-", ",", "(", ")", "."]:
        t = t.replace(symbol, " ")

    # Add a space at the start and end, so we can search for whole words
    t = " " + t + " "

    # Check keywords in order. The first match wins.
    if "machine learning" in t or " ml " in t or " mlops " in t:
        return "ML Engineer"
    if " ai " in t or " al " in t or "artificial intelligence" in t or " llm " in t:
        return "AI Engineer"
    if "data scien" in t or "applied scien" in t:
        return "Data Scientist"
    if "data engineer" in t or "data architect" in t or " etl " in t:
        return "Data Engineer"
    if "business analyst" in t:
        return "Business Analyst"
    if "analyst" in t or "analytics" in t or "business intelligence" in t:
        return "Data Analyst"
    return "Other"


# Apply the function to every title and store the result in a new column
clean["role"] = clean["title"].apply(get_role)

# Show how many jobs are in each role
print(clean["role"].value_counts())


role
AI Engineer         431
Data Analyst        405
Data Engineer       381
Business Analyst    348
ML Engineer         335
Data Scientist      309
Other               105
Name: count, dtype: int64


In [16]:
# Remove the column that is 93% empty (errors="ignore" means: don't complain if it's already gone)
clean = clean.drop(columns=["contract_type"], errors="ignore")

# Fill blanks in contract_time with a clear label
clean["contract_time"] = clean["contract_time"].fillna("Not specified")

# Save the clean table into the "processed" folder
clean.to_csv("../data/processed/jobs_clean.csv", index=False)

print("Saved! Rows and columns:", clean.shape)

Saved! Rows and columns: (2314, 23)
